In [1]:
# Author : Raghav Gupta
#  ============================================================
# Cell 1: Imports, data preparation check, and configuration
# ============================================================
# This cell imports required libraries, checks whether the prepared
# StackSats BTC analytics dataset exists, prepares it if missing,
# and defines all strategy settings.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import subprocess
from itertools import product

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy


# ============================================================
# StackSats prepared dataset check
# ============================================================
# This block prepares the file if it is missing.

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"


raw_brk_path = Path("../data/raw/brk_metrics.parquet")

if not btc_path.exists():
    print(f"Prepared dataset not found at: {btc_path}")
    print("Preparing StackSats analytics dataset...")

    if not raw_brk_path.exists():
        raise FileNotFoundError(
            f"Raw BRK metrics file not found at: {raw_brk_path}. "
            "Please update raw_brk_path to the correct location of brk_metrics.parquet."
        )

    subprocess.run(
        [
            "stacksats",
            "data",
            "prepare",
            "--source",
            str(raw_brk_path),
        ],
        check=True,
    )

if not btc_path.exists():
    raise FileNotFoundError(
        f"Failed to create prepared dataset at {btc_path}."
    )

print(f"Using prepared dataset: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per calendar-year evaluation window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Minimum number of rows required for a valid calendar-year evaluation window.
# Normal years have 365 rows; leap years have 366 rows and are kept fully.
WINDOW_SIZE = 365

# Default lookbacks used before grid search selects the best combination.
# These values are overwritten later by the best-performing grid result.
MOMENTUM_LOOKBACK = 45
SMA_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Grid values to test.
MOMENTUM_LOOKBACK_GRID = [30, 45, 60, 90]
SMA_LOOKBACK_GRID = [60, 90, 120, 200]
REGIME_LOOKBACK_GRID = [60, 90, 120, 200]

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Fallback strategy options used if a regime appears in test but was not seen in training.
# The grid search will test each fallback option for every lookback combination.
# The SMA fallback is dynamic because the column depends on the selected SMA lookback.
FALLBACK_STRATEGY_GRID = [
    "sma",
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
]

def resolve_fallback_strategy(fallback_strategy, sma_lookback=None):
    """Convert a fallback strategy option into the actual weight column name."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy == "sma":
        return f"sma_{sma_lookback}d_weight"

    return fallback_strategy

def get_fallback_strategy_cols(sma_lookback=None):
    """Return actual fallback strategy weight columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    return [
        resolve_fallback_strategy(fallback_strategy, sma_lookback)
        for fallback_strategy in FALLBACK_STRATEGY_GRID
    ]

# Default fallback strategy before grid search overwrites it.
FALLBACK_STRATEGY = resolve_fallback_strategy("sma", SMA_LOOKBACK)

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
def get_candidate_cols(sma_lookback=None):
    """Return candidate strategy columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    return [
        "stacksats_mvrv_weight",
        "stacksats_momentum_weight",
        f"sma_{sma_lookback}d_weight",
    ]

CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)


Using prepared dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet


In [2]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """

    # If improvement is greater than tolerance, strategy is better.
    if pct_diff > tolerance:
        return "better"

    # If improvement is below negative tolerance, strategy is worse.
    if pct_diff < -tolerance:
        return "worse"

    # Otherwise treat it as no meaningful difference.
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name. Positive values use a green upward
        arrow, while negative values use a red downward arrow.
    """

    # Positive result: green upward arrow.
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"

    # Negative result: red downward arrow.
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def get_calendar_year_windows(
    df: pd.DataFrame,
    split_start: str,
    split_end: str,
    min_days: int = WINDOW_SIZE,
):
    """
    Build one evaluation window per calendar year.

    This is used instead of fixed 365-row chunking so leap years are handled
    correctly. For example, the 2024 test window is 2024-01-01 to 2024-12-31
    with 366 rows, and the next test window starts on 2025-01-01.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with a datetime `date` column.
    split_start : str
        Inclusive split start date, such as TRAIN_START or TEST_START.
    split_end : str
        Inclusive split end date, such as TRAIN_END or TEST_END.
    min_days : int
        Minimum rows required to keep a calendar-year window. The default is
        365, so normal years and leap years are both valid.

    Returns
    -------
    tuple[pd.DataFrame, list[dict], pd.DataFrame, pd.DataFrame]
        Raw split dataframe, list of calendar-year window dictionaries,
        metadata for kept windows, and metadata for skipped years.
    """

    split_start_ts = pd.to_datetime(split_start)
    split_end_ts = pd.to_datetime(split_end)

    raw_split = (
        df[
            (df["date"] >= split_start_ts) &
            (df["date"] <= split_end_ts)
        ]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    windows = []
    kept_rows = []
    skipped_rows = []

    for year in range(split_start_ts.year, split_end_ts.year + 1):
        year_start = max(pd.Timestamp(year=year, month=1, day=1), split_start_ts)
        year_end = min(pd.Timestamp(year=year, month=12, day=31), split_end_ts)

        year_df = (
            raw_split[
                (raw_split["date"] >= year_start) &
                (raw_split["date"] <= year_end)
            ]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        observed_days = len(year_df)
        expected_days = (year_end - year_start).days + 1

        if observed_days < min_days:
            skipped_rows.append({
                "year": year,
                "start_date": year_start,
                "end_date": year_end,
                "observed_days": observed_days,
                "expected_calendar_days": expected_days,
                "reason": f"less than {min_days} rows",
            })
            continue

        window_number = len(windows) + 1
        windows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "data": year_df,
        })

        kept_rows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "is_leap_window": observed_days == 366,
        })

    window_metadata_df = pd.DataFrame(kept_rows)
    skipped_metadata_df = pd.DataFrame(skipped_rows)

    return raw_split, windows, window_metadata_df, skipped_metadata_df


def concat_calendar_windows(windows):
    """Concatenate the kept calendar-year windows into one dataframe."""

    if not windows:
        return pd.DataFrame()

    return pd.concat(
        [w["data"].copy() for w in windows],
        ignore_index=True,
    )

def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """

    # Convert input into a numpy array.
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    # Prevent empty signal arrays.
    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    # Replace NaN and infinity values with 0.
    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Apply a small floor so no value is exactly zero or negative.
    clean_signal = np.maximum(clean_signal, signal_floor)

    # If all signals are invalid or zero, fall back to uniform weights.
    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    # Normalize so all daily weights sum to 1.
    return clean_signal / clean_signal.sum()


def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """

    # Number of full 365-day windows.
    n_windows = len(window_summary_df)

    # Sum strategy and DCA sats-per-dollar across windows.
    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()

    # Extra sats-per-dollar gained or lost vs DCA.
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    # Ratio of strategy SPD to DCA SPD.
    spd_ratio = strategy_spd_sum / dca_spd_sum

    # Percentage improvement over DCA.
    improvement_pct = (spd_ratio - 1.0) * 100.0

    # Sum total sats accumulated by strategy and DCA.
    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()

    # Extra sats accumulated vs DCA.
    extra_sats = strategy_sats - dca_sats

    # Count windows where strategy beat, lost to, or tied DCA.
    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    # Window win rate.
    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    # Return one summary dictionary.
    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """

    # Actual BTC price values used as tick positions.
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    # Labels displayed on the chart.
    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext


In [3]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [4]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV, Momentum, and Uniform strategies.
#
# Calendar-year export method used here:
# 1. Export one calendar year at a time: Jan 1 to Dec 31.
# 2. Ask StackSats to export that year.
# 3. Keep the latest export window inside that year using max(end_date).
# 4. Use the exported dates as the valid dates for that year.
#
# Leap-year behavior:
# For a leap year such as 2024, the input date range is still
# 2024-01-01 to 2024-12-31. However, if StackSats internally produces
# 365-day export windows, the latest export window may contain 365 rows
# such as 2024-01-02 to 2024-12-31.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
    "stacksats_uniform_weight": UniformStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weight_frame_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one calendar-year window.

    After StackSats export, it keeps only the rows whose export end_date
    equals the latest end_date in that calendar year.

    Returns
    -------
    pd.DataFrame
        Columns: date, raw_weight
    """

    # Get current window start and end dates.
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    # Cache key is based on strategy and date range.
    cache_key = (strategy_key, window_start, window_end, "latest_end_window")

    # If already computed, return cached version.
    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    # Create StackSats export config for this calendar window.
    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    # Filter BTC data to the current calendar-year window only.
    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    # Stop if no data exists for this window.
    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    # Select the requested StackSats strategy object.
    strategy = stacksats_strategy_objects[strategy_key]

    # Run StackSats export.
    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    # Convert export result to dataframe.
    weights = export_obj.to_dataframe()

    # Ensure output is a Polars dataframe.
    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    # Cast date columns into datetime format.
    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    # Keep only the latest export window
    latest_end = weights.select(pl.col("end_date").max()).item()

    latest_window_weights = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .select(["date", "weight"])
        .sort("date")
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    latest_window_weights["date"] = pd.to_datetime(latest_window_weights["date"])

    # Keep only dates inside the requested calendar-year range.
    start_ts = pd.to_datetime(window_start)
    end_ts = pd.to_datetime(window_end)
    latest_window_weights = latest_window_weights[
        (latest_window_weights["date"] >= start_ts) &
        (latest_window_weights["date"] <= end_ts)
    ].copy()

    if latest_window_weights.empty:
        raise ValueError(
            f"StackSats export returned no usable latest-window weights for {strategy_key} "
            f"from {window_start} to {window_end}."
        )

    # If duplicate dates exist, keep the last one after sorting.
    latest_window_weights = (
        latest_window_weights
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    # Store in cache.
    _export_cache[cache_key] = latest_window_weights.copy()

    return latest_window_weights.copy()


In [5]:
# ============================================================
# Cell 5: Feature engineering and simplified regime classification
# ============================================================
# This cell creates reusable functions for:
# 1. engineering rolling lookback features
# 2. assigning the simpler combined regime
# 3. preparing a clean dataframe for any lookback combination

def classify_btc_mvrv_market_cap_regime(
    row,
    momentum_lookback=None,
    sma_lookback=None,
    regime_lookback=None,
):
    """
    Classify each day into a simpler BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime using SMA ratio and returns
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    if momentum_lookback is None:
        momentum_lookback = MOMENTUM_LOOKBACK
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    if regime_lookback is None:
        regime_lookback = REGIME_LOOKBACK

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    sma_selected_ratio = row[f"price_{sma_lookback}d_sma_ratio"]
    sma_regime_ratio = row[f"price_{regime_lookback}d_sma_ratio"]
    return_momentum = row[f"btc_return_{momentum_lookback}d"]
    return_sma = row[f"btc_return_{sma_lookback}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime without drawdown
    # ------------------------------------------------------------
    if sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime


def prepare_btc_data_for_lookbacks(
    full_btc_df,
    momentum_lookback,
    sma_lookback,
    regime_lookback,
):
    """
    Create rolling features and simplified regimes for one lookback combination.
    """

    lookback_days = sorted({
        momentum_lookback,
        sma_lookback,
        regime_lookback,
    })

    feature_exprs = []

    for d in lookback_days:
        feature_exprs.extend([
            pl.col("price_usd")
            .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
            .alias(f"price_{d}d_sma"),

            pl.col("price_usd")
            .pct_change(d)
            .alias(f"btc_return_{d}d"),
        ])

    temp_btc_df = full_btc_df.with_columns(feature_exprs)

    ratio_exprs = []

    for d in lookback_days:
        ratio_exprs.append(
            (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
            .alias(f"price_{d}d_sma_ratio")
        )

    temp_btc_df = temp_btc_df.with_columns(ratio_exprs)

    temp_btc_data = temp_btc_df.to_pandas()
    temp_btc_data["date"] = pd.to_datetime(temp_btc_data["date"])

    feature_cols = [
        "price_usd",
        "mvrv",
        "realized_cap_growth_rate",
        "market_cap_growth_rate",
    ]

    for d in lookback_days:
        feature_cols.extend([
            f"price_{d}d_sma",
            f"price_{d}d_sma_ratio",
            f"btc_return_{d}d",
        ])

    temp_btc_data = (
        temp_btc_data
        .dropna(subset=feature_cols)
        .sort_values("date")
        .reset_index(drop=True)
    )

    temp_btc_data["combined_regime"] = temp_btc_data.apply(
        lambda row: classify_btc_mvrv_market_cap_regime(
            row,
            momentum_lookback=momentum_lookback,
            sma_lookback=sma_lookback,
            regime_lookback=regime_lookback,
        ),
        axis=1,
    )

    return temp_btc_data


In [10]:
# ============================================================
# Cell 6: Candidate strategy weights
# ============================================================
# This cell creates daily weights for each candidate strategy.
# Candidate strategies are StackSats MVRV, StackSats Momentum, and custom SMA.
#
# Important:
# StackSats strategies keep the latest export window for each calendar year.
# In leap years, that may be 365 rows instead of 366. To keep all strategies
# comparable, this function evaluates DCA, SMA, MVRV, and Momentum on the
# common exported dates for that year.

def create_candidate_strategy_weights_simple(data, sma_lookback=None):
    """
    Create daily allocation weights for all candidate strategies.

    Parameters
    ----------
    data : pd.DataFrame
        One calendar-year window of BTC data with regime and engineered features.
    sma_lookback : int or None
        SMA lookback used for the custom SMA candidate.

    Returns
    -------
    pd.DataFrame
        Original data plus strategy weight columns:
        - dca_weight
        - stacksats_mvrv_weight
        - stacksats_momentum_weight
        - sma_{sma_lookback}d_weight
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    # Sort data by date and reset index.
    df = data.copy().sort_values("date").reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])

    # Stop if the input window is empty.
    if len(df) == 0:
        raise ValueError("No data available.")

    # Export StackSats MVRV latest-window weights for this calendar year.
    mvrv_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_mvrv_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_mvrv_raw_weight"})

    # Export StackSats Momentum latest-window weights for this calendar year.
    momentum_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_momentum_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_momentum_raw_weight"})

    # Export StackSats UniformStrategy latest-window weights for this calendar year.
    uniform_weights = export_stacksats_weight_frame_for_window(
        strategy_key="stacksats_uniform_weight",
        window_df=df,
        full_btc_df=btc_df,
    ).rename(columns={"raw_weight": "stacksats_uniform_raw_weight"})

    # Keep only the common dates available in all StackSats exports.
    df = (
        df
        .merge(mvrv_weights, on="date", how="inner")
        .merge(momentum_weights, on="date", how="inner")
        .merge(uniform_weights, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # Number of usable days in the current calendar-year export.
    n = len(df)
    if n < WINDOW_SIZE:
        raise ValueError(
            f"Only {n} common StackSats export dates were available; "
            f"expected at least {WINDOW_SIZE}."
        )

    # Uniform DCA benchmark from StackSats UniformStrategy.
    # Normalize the exported raw uniform weights over the same common dates
    df["dca_weight"] = build_simple_normalized_weights(
        df["stacksats_uniform_raw_weight"].values
    )

    # Normalize StackSats raw weights so each candidate sums to 1 over the
    # same evaluated dates.
    df["stacksats_mvrv_weight"] = build_simple_normalized_weights(
        df["stacksats_mvrv_raw_weight"].values
    )

    df["stacksats_momentum_weight"] = build_simple_normalized_weights(
        df["stacksats_momentum_raw_weight"].values
    )

    # Candidate 3: Custom SMA strategy.
    # price/SMA ratio < 1 means BTC price is below the selected SMA.
    # In that case, sma_signal becomes positive and allocation increases.
    sma_signal = (1.0 - df[f"price_{sma_lookback}d_sma_ratio"]).clip(-1, 1)

    # Convert SMA signal into a multiplier.
    # 1.50 controls how strongly the SMA signal changes allocation.
    sma_multiplier = np.maximum(SIGNAL_FLOOR, 1.0 + 1.50 * sma_signal)

    # Normalize SMA multipliers into daily weights that sum to 1 over the
    # same dates used by StackSats exports.
    df[f"sma_{sma_lookback}d_weight"] = build_simple_normalized_weights(
        sma_multiplier.values
    )

    # Keep raw export columns for debugging, but they are not used for strategy selection.
    return df


In [11]:
# ============================================================
# Cell 7: Calendar-year evaluation functions for one lookback combination
# ============================================================
# This cell evaluates candidate strategies by regime, learns the best mapping,
# and applies that mapping to train/test calendar-year windows.
#
# Leap-year handling:
# - Calendar years are created separately for train and test.
# - For example, 2024 is requested as 2024-01-01 to 2024-12-31.
# - Because StackSats may return rolling 365-day export windows, the latest
#   export window inside a leap year may contain 365 rows, such as
#   2024-01-02 to 2024-12-31.
# - The notebook keeps the export rows with the latest end_date

def evaluate_strategies_by_regime_in_365_windows(
    windows,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
):
    """
    Evaluate all candidate strategies inside each regime for every calendar-year window.

    Input is a list of calendar-year windows produced by get_calendar_year_windows().
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    candidate_cols = get_candidate_cols(sma_lookback)
    rows = []

    for window_info in windows:
        window_idx = window_info["window"]
        year = window_info["year"]
        window_data = window_info["data"].copy().reset_index(drop=True)

        df = create_candidate_strategy_weights_simple(
            data=window_data,
            sma_lookback=sma_lookback,
        )

        for regime, regime_df in df.groupby("combined_regime"):
            if len(regime_df) < MIN_REGIME_DAYS:
                continue

            dca_sats = (
                regime_df["dca_weight"]
                * total_budget_usd
                / regime_df["price_usd"]
                * 100_000_000
            ).sum()

            for col in candidate_cols:
                strategy_sats = (
                    regime_df[col]
                    * total_budget_usd
                    / regime_df["price_usd"]
                    * 100_000_000
                ).sum()

                strategy_spd = strategy_sats / total_budget_usd
                dca_spd = dca_sats / total_budget_usd
                extra_sats = strategy_sats - dca_sats
                extra_spd = strategy_spd - dca_spd
                spd_ratio = strategy_spd / dca_spd
                improvement_pct = (spd_ratio - 1.0) * 100.0

                rows.append({
                    "train_window": window_idx,
                    "year": year,
                    "window_start_date": window_info["start_date"],
                    "window_end_date": window_info["end_date"],
                    "window_days": window_info["days"],
                    "combined_regime": regime,
                    "days": len(regime_df),
                    "strategy": col,
                    "strategy_sats": strategy_sats,
                    "dca_sats": dca_sats,
                    "extra_sats_vs_dca": extra_sats,
                    "strategy_spd": strategy_spd,
                    "dca_spd": dca_spd,
                    "extra_spd_vs_dca": extra_spd,
                    "spd_ratio": spd_ratio,
                    "improvement_pct": improvement_pct,
                    "status": get_status_from_pct_diff(improvement_pct),
                })

    return pd.DataFrame(rows)


def learn_best_mapping_by_regime(train_regime_results_df):
    """
    Learn the best candidate strategy for each regime using training data only.
    """

    best_mapping_df = (
        train_regime_results_df
        .groupby(["combined_regime", "strategy"], as_index=False)
        .agg(
            total_days=("days", "sum"),
            mean_improvement_pct=("improvement_pct", "mean"),
            median_improvement_pct=("improvement_pct", "median"),
            mean_extra_spd_vs_dca=("extra_spd_vs_dca", "mean"),
            total_extra_sats_vs_dca=("extra_sats_vs_dca", "sum"),
            windows_seen=("train_window", "nunique"),
        )
    )

    best_mapping_df = (
        best_mapping_df
        .sort_values(
            ["combined_regime", "mean_improvement_pct", "total_days"],
            ascending=[True, False, False],
        )
        .groupby("combined_regime")
        .head(1)
        .reset_index(drop=True)
    )

    best_mapping_df["status"] = best_mapping_df["mean_improvement_pct"].apply(
        get_status_from_pct_diff
    )

    return best_mapping_df


def apply_regime_mapping_to_one_window(
    window_data,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    window_number=None,
    year=None,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to one calendar-year window.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    df = create_candidate_strategy_weights_simple(
        data=window_data,
        sma_lookback=sma_lookback,
    )

    regime_to_strategy = dict(
        zip(best_mapping_df["combined_regime"], best_mapping_df["strategy"])
    )

    mapped_strategy = df["combined_regime"].map(regime_to_strategy)
    df["used_fallback_strategy"] = mapped_strategy.isna()
    df["fallback_strategy"] = fallback_strategy
    df["selected_strategy"] = mapped_strategy.fillna(fallback_strategy)

    df["raw_selected_weight"] = df.apply(
        lambda row: row[row["selected_strategy"]],
        axis=1,
    )

    raw_weight_sum = df["raw_selected_weight"].sum()
    if raw_weight_sum <= 0 or pd.isna(raw_weight_sum):
        df["final_strategy_weight"] = 1.0 / len(df)
    else:
        df["final_strategy_weight"] = df["raw_selected_weight"] / raw_weight_sum

    df["final_strategy_usd"] = df["final_strategy_weight"] * total_budget_usd
    df["dca_usd"] = df["dca_weight"] * total_budget_usd

    df["btc_accum_strategy"] = df["final_strategy_usd"] / df["price_usd"]
    df["btc_accum_dca"] = df["dca_usd"] / df["price_usd"]

    df["sats_accum_strategy"] = df["btc_accum_strategy"] * 100_000_000
    df["sats_accum_dca"] = df["btc_accum_dca"] * 100_000_000

    df["strategy_spd_daily"] = df["sats_accum_strategy"] / total_budget_usd
    df["dca_spd_daily"] = df["sats_accum_dca"] / total_budget_usd

    if window_number is not None:
        df["window"] = window_number
    if year is not None:
        df["year"] = year

    return df


def apply_regime_mapping_to_window_set(
    windows,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    sma_lookback=None,
    fallback_strategy=None,
):
    """
    Apply the learned regime mapping to every calendar-year window.
    """

    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    window_dfs = []
    window_summary_rows = []

    for window_info in windows:
        window_strategy_df = apply_regime_mapping_to_one_window(
            window_data=window_info["data"].copy().reset_index(drop=True),
            best_mapping_df=best_mapping_df,
            total_budget_usd=total_budget_usd,
            window_number=window_info["window"],
            year=window_info["year"],
            sma_lookback=sma_lookback,
            fallback_strategy=fallback_strategy,
        )

        strategy_sats = window_strategy_df["sats_accum_strategy"].sum()
        dca_sats = window_strategy_df["sats_accum_dca"].sum()

        strategy_spd = strategy_sats / total_budget_usd
        dca_spd = dca_sats / total_budget_usd

        extra_sats = strategy_sats - dca_sats
        extra_spd = strategy_spd - dca_spd

        spd_ratio = strategy_spd / dca_spd
        improvement_pct = (spd_ratio - 1.0) * 100.0

        window_summary_rows.append({
            "window": window_info["window"],
            "year": window_info["year"],
            "start_date": window_strategy_df["date"].min(),
            "end_date": window_strategy_df["date"].max(),
            "days": len(window_strategy_df),
            "expected_calendar_days": window_info["expected_calendar_days"],
            "is_leap_window": len(window_strategy_df) == 366,
            "budget_usd": total_budget_usd,
            "strategy_sats": strategy_sats,
            "dca_sats": dca_sats,
            "extra_sats_vs_dca": extra_sats,
            "strategy_spd": strategy_spd,
            "dca_spd": dca_spd,
            "extra_spd_vs_dca": extra_spd,
            "spd_ratio": spd_ratio,
            "improvement_pct": improvement_pct,
            "result": get_status_from_pct_diff(improvement_pct),
            "weight_sum": window_strategy_df["final_strategy_weight"].sum(),
            "max_weight": window_strategy_df["final_strategy_weight"].max(),
            "min_weight": window_strategy_df["final_strategy_weight"].min(),
            "days_above_dca_weight": int(
                (window_strategy_df["final_strategy_weight"] > window_strategy_df["dca_weight"]).sum()
            ),
            "fallback_strategy": fallback_strategy,
            "fallback_days": int(window_strategy_df["used_fallback_strategy"].sum()),
        })

        window_dfs.append(window_strategy_df)

    if not window_dfs:
        raise ValueError("No valid calendar-year windows were available.")

    out_df = pd.concat(window_dfs, ignore_index=True)
    summary_df = pd.DataFrame(window_summary_rows)

    return out_df, summary_df


In [12]:
# ============================================================
# Cell 8: Run full calendar-year strategy for one lookback combination
# ============================================================
# This function is used by the grid search and again for the selected best combo.

def run_full_regime_strategy_for_lookbacks(
    momentum_lookback,
    sma_lookback,
    regime_lookback,
    fallback_strategy=None,
    total_budget_usd=TOTAL_BUDGET_USD,
):
    """
    Run the complete simple-regime strategy for one lookback combination.

    The best regime-to-strategy mapping is learned from training data only,
    then applied separately to train and test windows.
    """

    if fallback_strategy is None:
        fallback_strategy = FALLBACK_STRATEGY

    fallback_strategy = resolve_fallback_strategy(
        fallback_strategy=fallback_strategy,
        sma_lookback=sma_lookback,
    )

    # Prepare features and simplified regime labels for this combination.
    local_btc_data = prepare_btc_data_for_lookbacks(
        full_btc_df=btc_df,
        momentum_lookback=momentum_lookback,
        sma_lookback=sma_lookback,
        regime_lookback=regime_lookback,
    )

    # Train/test split using calendar-year windows.
    # This avoids fixed 365-row chunking across leap years.
    raw_train, local_train_windows, train_calendar_windows_df, train_skipped_years_df = get_calendar_year_windows(
        local_btc_data,
        TRAIN_START,
        TRAIN_END,
        min_days=WINDOW_SIZE,
    )

    raw_test, local_test_windows, test_calendar_windows_df, test_skipped_years_df = get_calendar_year_windows(
        local_btc_data,
        TEST_START,
        TEST_END,
        min_days=WINDOW_SIZE,
    )

    local_train_eval_df = concat_calendar_windows(local_train_windows)
    local_test_eval_df = concat_calendar_windows(local_test_windows)

    local_n_train_windows = len(local_train_windows)
    local_n_test_windows = len(local_test_windows)

    if local_n_train_windows == 0 or local_n_test_windows == 0:
        raise ValueError(
            "Not enough data to create train and test calendar-year windows for this lookback combination."
        )

    # Evaluate candidates by regime on training calendar years only.
    local_train_regime_results_df = evaluate_strategies_by_regime_in_365_windows(
        local_train_windows,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
    )

    if local_train_regime_results_df.empty:
        raise ValueError("No regime-level training results were created.")

    # Learn best strategy per regime from training only.
    local_best_mapping_df = learn_best_mapping_by_regime(
        local_train_regime_results_df
    )

    # Apply learned mapping to train and test periods.
    local_train_strategy_df, local_train_window_summary_df = apply_regime_mapping_to_window_set(
        local_train_windows,
        local_best_mapping_df,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
        fallback_strategy=fallback_strategy,
    )

    local_test_strategy_df, local_test_window_summary_df = apply_regime_mapping_to_window_set(
        local_test_windows,
        local_best_mapping_df,
        total_budget_usd=total_budget_usd,
        sma_lookback=sma_lookback,
        fallback_strategy=fallback_strategy,
    )

    # Summarize train and test performance using the same SPD logic as the final chart.
    local_train_spd_summary = summarize_spd_like_composite(
        local_train_window_summary_df
    )
    local_test_spd_summary = summarize_spd_like_composite(
        local_test_window_summary_df
    )

    local_split_summary_df = pd.DataFrame([
        {
            "split": "train",
            "start_date": raw_train["date"].min(),
            "end_date": raw_train["date"].max(),
            "rows_total": len(raw_train),
            "rows_eval": len(local_train_eval_df),
            "windows": local_n_train_windows,
            "calendar_window_start_dates": ", ".join(train_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")),
            "calendar_window_end_dates": ", ".join(train_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")),
            "skipped_years": ", ".join(train_skipped_years_df["year"].astype(str)) if not train_skipped_years_df.empty else "",
            "budget_rule": "$1,000 per calendar-year training window; StackSats uses latest export window per year",
        },
        {
            "split": "test",
            "start_date": raw_test["date"].min(),
            "end_date": raw_test["date"].max(),
            "rows_total": len(raw_test),
            "rows_eval": len(local_test_eval_df),
            "windows": local_n_test_windows,
            "calendar_window_start_dates": ", ".join(test_calendar_windows_df["start_date"].dt.strftime("%Y-%m-%d")),
            "calendar_window_end_dates": ", ".join(test_calendar_windows_df["end_date"].dt.strftime("%Y-%m-%d")),
            "skipped_years": ", ".join(test_skipped_years_df["year"].astype(str)) if not test_skipped_years_df.empty else "",
            "budget_rule": "$1,000 per calendar-year test window; StackSats uses latest export window per year",
        },
    ])

    return {
        "momentum_lookback": momentum_lookback,
        "sma_lookback": sma_lookback,
        "regime_lookback": regime_lookback,
        "fallback_strategy": fallback_strategy,
        "btc_data": local_btc_data,
        "split_summary_df": local_split_summary_df,
        "train_calendar_windows_df": train_calendar_windows_df,
        "test_calendar_windows_df": test_calendar_windows_df,
        "train_skipped_years_df": train_skipped_years_df,
        "test_skipped_years_df": test_skipped_years_df,
        "train_eval_df": local_train_eval_df,
        "test_eval_df": local_test_eval_df,
        "train_regime_results_df": local_train_regime_results_df,
        "best_mapping_df": local_best_mapping_df,
        "train_strategy_df": local_train_strategy_df,
        "test_strategy_df": local_test_strategy_df,
        "train_window_summary_df": local_train_window_summary_df,
        "test_window_summary_df": local_test_window_summary_df,
        "train_spd_summary": local_train_spd_summary,
        "test_spd_summary": local_test_spd_summary,
    }
